In [1]:
import pandas as pd
import numpy as np

In [2]:
# Read input data from FRBNY website
per_gdp =  pd.read_excel("inputData/per_data.xlsx", sheet_name="GDP")
per_gdp = per_gdp.set_index("Date")
per_gdp['gdp.log'] = np.log(per_gdp['GDP'])
#drop gdp raw 
per_gdp = per_gdp.drop(columns = ['GDP'])
per_gdp






,gdp.log
Date,
1979-01-01,10.449106
1979-04-01,10.618445
1979-07-01,10.613261
1979-10-01,10.649778
1980-01-01,10.501155
...,...
2023-10-01,11.913979
2024-01-01,11.811955
2024-04-01,11.888286


In [3]:
per_inflation =  pd.read_excel("inputData/per_data.xlsx", sheet_name="inflation")
per_inflation = per_inflation.set_index("Date")
per_inflation['index'] = np.nan
#create index from monthly increase 
#set first row to 100
# per_inflation['index'].iloc[0] = 100
# per_inflation
# for i in range(1,len(per_inflation)):
#     per_inflation['index'][i] = ((per_inflation['inflation.mensual'][i]/100) * per_inflation['index'][i-1]) + per_inflation['index'][i-1]

#resample 'index'column to quarterly
per_inflation = per_inflation['inflation'].resample('QS').mean()
per_inflation['growth'] = per_inflation.pct_change() * 100
per_inflation['growth'] = per_inflation['growth'].dropna()

#annualize quarterly growth
per_inflation['annualized'] =  100 * ((1 + per_inflation['growth']/100)**4 - 1)
per_inflation['annualized']
per_inflation_final = per_inflation['annualized'].to_frame()
per_inflation_final['expectations'] = per_inflation_final.rolling(4).mean()
per_inflation_final = per_inflation_final.dropna()
per_inflation_final


,inflation,expectations
Date,,
1993-01-01,54.967510,53.653094
1993-04-01,48.993463,50.470253
1993-07-01,34.362353,47.844190
1993-10-01,27.433384,41.439177
1994-01-01,24.078473,33.716918
...,...,...
2024-01-01,3.276685,4.175504
2024-04-01,3.967937,3.425259
2024-07-01,1.932585,2.931913


In [4]:
#.interest    
per_interest =  pd.read_excel("inputData/per_data.xlsx", sheet_name="reference")
per_interest = per_interest.set_index("Date")
per_interest = per_interest.resample('QS').mean()
per_interest['interest'] = per_interest['interest'].dropna()
per_interest

,interest
Date,
2003-07-01,2.750000
2003-10-01,2.583333
2004-01-01,2.500000
2004-04-01,2.500000
2004-07-01,2.666667
...,...
2024-01-01,6.333333
2024-04-01,5.833333
2024-07-01,5.500000


In [5]:
per_ib = pd.read_excel("inputData/per_data.xlsx", sheet_name="interbank")
per_ib = per_ib.set_index("Date")
per_ib = per_ib.resample('QS').mean()
per_ib['interest'] = per_ib['interest'].dropna()
per_ib

,interest
Date,
1995-10-01,9.643333
1996-01-01,11.240000
1996-04-01,12.743333
1996-07-01,12.423333
1996-10-01,16.020000
...,...
2024-01-01,6.399233
2024-04-01,5.892167
2024-07-01,5.529500


In [6]:
per_ov = pd.read_excel("inputData/per_data.xlsx", sheet_name="overnight")
per_ov = per_ov.set_index("Date")
per_ov = per_ov.resample('QS').mean()
per_ov['interest'] = per_ov['interest'].dropna()
per_ov

,interest
Date,
2001-01-01,4.760000
2001-04-01,4.000000
2001-07-01,4.000000
2001-10-01,2.500000
2002-01-01,1.933333
...,...
2024-01-01,3.750000
2024-04-01,3.583333
2024-07-01,3.416667


In [7]:
# per_interest = per_interest
per_interest = per_ib
#per_interest = per_ov
per_interest

,interest
Date,
1995-10-01,9.643333
1996-01-01,11.240000
1996-04-01,12.743333
1996-07-01,12.423333
1996-10-01,16.020000
...,...
2024-01-01,6.399233
2024-04-01,5.892167
2024-07-01,5.529500


In [8]:
#merge per_gdp, per_inflation_final, per_interest
per_data = pd.concat([per_gdp, per_inflation_final, per_interest], axis=1).dropna()
per_data

,gdp.log,inflation,expectations,interest
Date,,,,
1995-10-01,10.794344,8.534320,10.938627,9.643333
1996-01-01,10.776550,9.190898,10.265602,11.240000
1996-04-01,10.857338,13.358166,10.296562,12.743333
1996-07-01,10.821229,8.085779,9.792291,12.423333
1996-10-01,10.842291,7.875346,9.627547,16.020000
...,...,...,...,...
2023-10-01,11.913979,2.550446,4.794620,7.076100
2024-01-01,11.811955,3.276685,4.175504,6.399233
2024-04-01,11.888286,3.967937,3.425259,5.892167


In [9]:
#convert index to Q1 
per_data.index = per_data.index.to_period('Q')

stringency = pd.read_csv("inputData/stringency.csv")
stringency_data = stringency[['CountryCode', 'Date', 'StringencyIndex_Average']]
stringency_data = stringency_data[stringency_data['CountryCode'] == 'PER']
stringency_data = stringency_data.drop(columns=['CountryCode'])
# stringency_data['Date'] = pd.to_datetime(stringency_data['Date'])
stringency_data = stringency_data.set_index('Date')
#process 20200101 to 2020-01-01
stringency_data.index = pd.to_datetime(stringency_data.index, format='%Y%m%d')
stringency_data = stringency_data.resample('QS').mean()
stringency_data = stringency_data.rename(columns={'StringencyIndex_Average': 'covid.indicator'})
stringency_data.index = stringency_data.index.to_period('Q')
#add rows up to 2024Q1 with stringency declining linearly to 0
stringency_data = stringency_data.reindex(pd.period_range(start='2020Q1', end='2024Q1', freq='Q', name='Date'))
stringency_data.loc['2024Q1'] = 0
stringency_data = stringency_data.interpolate(method='linear')
#add rows forom 1990Q1 to 2019Q4 with stringency = 0
stringency_data = stringency_data.reindex(pd.period_range(start='1990Q1', end='2024Q1', freq='Q', name='Date'))
stringency_data = stringency_data.fillna(0)
stringency_data = stringency_data.dropna()
stringency_data

,covid.indicator
Date,
1990Q1,0.000000
1990Q2,0.000000
1990Q3,0.000000
1990Q4,0.000000
1991Q1,0.000000
...,...
2023Q1,11.610087
2023Q2,8.707565
2023Q3,5.805043


In [10]:
#MERGE DATA
per_data = per_data.merge(stringency_data, left_index=True, right_index=True, how='left')
per_data = per_data.dropna()
per_data

,gdp.log,inflation,expectations,interest,covid.indicator
Date,,,,,
1995Q4,10.794344,8.534320,10.938627,9.643333,0.000000
1996Q1,10.776550,9.190898,10.265602,11.240000,0.000000
1996Q2,10.857338,13.358166,10.296562,12.743333,0.000000
1996Q3,10.821229,8.085779,9.792291,12.423333,0.000000
1996Q4,10.842291,7.875346,9.627547,16.020000,0.000000
...,...,...,...,...,...
2023Q1,11.798235,5.753149,6.662561,7.713567,11.610087
2023Q2,11.852270,6.968915,6.489917,7.736100,8.707565
2023Q3,11.868844,3.905969,5.647934,7.698567,5.805043


In [11]:
#rename columns
per_data.columns = ['gdp.log', 'inflation', 'inflation.expectations', 'interest', 'covid.indicator']

#add columns of 0 called covid.indicator
#per_data['covid.indicator'] = 0

#save data to excel
per_data.to_excel("inputData/Holston_Laubach_Williams_PER.xlsx", sheet_name="PER Input Data" )
